In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

In [3]:
#load data
train = pd.read_csv(r"C:\Users\hecto\Documents\PythonProjects\obesity_risk\train.csv")

test = pd.read_csv(r"C:\Users\hecto\Documents\PythonProjects\obesity_risk\test.csv")

In [4]:
train.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [5]:
train.columns

Index(['id', 'Gender', 'Age', 'Height', 'Weight',
       'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC',
       'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad'],
      dtype='str')

In [7]:
#Define Target and Predictors
target = "NObeyesdad"

X = train.drop(columns=["id", target])

y = train[target]

X_test = test.drop(columns=["id"])


In [8]:
#Encode Target Variable
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

In [9]:
#Identify Categorical and Numeric Variables
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Columns:")
print(categorical_cols)

print("\nNumeric Columns:")
print(numeric_cols)

Categorical Columns:
['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

Numeric Columns:
['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']


C:\Users\hecto\AppData\Local\Temp\ipykernel_16980\2499057334.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [11]:
#train/test split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

In [12]:
#processing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [13]:
#Multinomial Logistic Regression
log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        multi_class="multinomial",
        max_iter=1000,
        random_state=42
    ))
])

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_val)

print("Logistic Regression Accuracy:")
print(accuracy_score(y_val, log_pred))

C:\ProgramData\anaconda3\envs\ds_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression Accuracy:
0.8692196531791907


In [14]:
print(classification_report(
    y_val,
    log_pred,
    target_names=label_encoder.classes_
))

                     precision    recall  f1-score   support

Insufficient_Weight       0.89      0.95      0.92       505
      Normal_Weight       0.87      0.82      0.85       617
     Obesity_Type_I       0.81      0.85      0.83       582
    Obesity_Type_II       0.93      0.96      0.95       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.75      0.71      0.73       485
Overweight_Level_II       0.73      0.71      0.72       504

           accuracy                           0.87      4152
          macro avg       0.85      0.86      0.85      4152
       weighted avg       0.87      0.87      0.87      4152



The multinomial logistic regression model achieved strong classification performance, producing an overall validation accuracy of approximately 87%. Several obesity categories were classified with high precision and recall, particularly Obesity_Type_III, which achieved perfect classification performance. The model also performed well for Insufficient_Weight, Normal_Weight, and Obesity_Type_II categories.

The weakest performance occurred among the Overweight_Level_I and Overweight_Level_II categories, likely because these adjacent classes contain overlapping physical and behavioral characteristics. Overall, the results indicate that the predictor variables contain substantial information related to obesity risk classification and that multinomial logistic regression is highly effective for modeling this multi-class problem.

In [15]:
X_test

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,Male,26.899886,1.848294,120.644178,yes,yes,2.938616,3.000000,Sometimes,no,2.825629,no,0.855400,0.000000,Sometimes,Public_Transportation
1,Female,21.000000,1.600000,66.000000,yes,yes,2.000000,1.000000,Sometimes,no,3.000000,no,1.000000,0.000000,Sometimes,Public_Transportation
2,Female,26.000000,1.643355,111.600553,yes,yes,3.000000,3.000000,Sometimes,no,2.621877,no,0.000000,0.250502,Sometimes,Public_Transportation
3,Male,20.979254,1.553127,103.669116,yes,yes,2.000000,2.977909,Sometimes,no,2.786417,no,0.094851,0.000000,Sometimes,Public_Transportation
4,Female,26.000000,1.627396,104.835346,yes,yes,3.000000,3.000000,Sometimes,no,2.653531,no,0.000000,0.741069,Sometimes,Public_Transportation
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13835,Male,23.327836,1.721384,78.030383,yes,no,2.813234,3.000000,Sometimes,no,1.000000,no,0.807076,0.778632,Sometimes,Public_Transportation
13836,Female,29.000000,1.590000,62.000000,no,yes,3.000000,3.000000,Sometimes,no,2.000000,no,0.000000,0.000000,Sometimes,Public_Transportation
13837,Female,22.935612,1.585547,44.376637,no,yes,3.000000,2.273740,Frequently,no,2.000000,no,1.949840,1.000000,Sometimes,Public_Transportation
13838,Male,21.000000,1.620000,53.000000,yes,yes,2.000000,3.000000,Sometimes,no,2.000000,no,3.000000,2.000000,no,Public_Transportation


In [16]:
final_predictions = log_model.predict(X_test)

In [17]:
#Convert Encoded Labels Back to Original Classes
final_labels = label_encoder.inverse_transform(final_predictions)

In [18]:
#Build Submission DataFrame
submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": final_labels
})

submission.head()

,id,NObeyesdad
0,20758,Obesity_Type_II
1,20759,Overweight_Level_I
2,20760,Obesity_Type_III
3,20761,Obesity_Type_I
4,20762,Obesity_Type_III


In [19]:
submission.to_csv(
    "obesity_logistic_submission.csv",
    index=False
)

#MODEL 2 — LDA

In [23]:
#Process data for LDA
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Convert only if sparse
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()

if hasattr(X_val_processed, "toarray"):
    X_val_processed = X_val_processed.toarray()

if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

In [24]:
lda_model = LinearDiscriminantAnalysis()

lda_model.fit(X_train_processed, y_train)

,solver,'svd'
,shrinkage,None
,priors,None
,n_components,None
,store_covariance,False
,tol,0.0001
,covariance_estimator,None


In [28]:
#predict
lda_pred = lda_model.predict(X_val_processed)

In [29]:
#Evaluate
print("LDA Accuracy:")
print(accuracy_score(y_val, lda_pred))

print(classification_report(
    y_val,
    lda_pred,
    target_names=label_encoder.classes_
))

LDA Accuracy:
0.8229768786127167
                     precision    recall  f1-score   support

Insufficient_Weight       0.81      0.93      0.87       505
      Normal_Weight       0.79      0.72      0.75       617
     Obesity_Type_I       0.78      0.78      0.78       582
    Obesity_Type_II       0.91      0.94      0.93       650
   Obesity_Type_III       0.99      1.00      0.99       809
 Overweight_Level_I       0.68      0.60      0.64       485
Overweight_Level_II       0.66      0.68      0.67       504

           accuracy                           0.82      4152
          macro avg       0.80      0.81      0.80      4152
       weighted avg       0.82      0.82      0.82      4152



In [30]:
lda_test_pred = lda_model.predict(X_test_processed)

lda_labels = label_encoder.inverse_transform(lda_test_pred)

lda_submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": lda_labels
})

lda_submission.to_csv("obesity_lda_submission.csv", index=False)

lda_submission.head()

,id,NObeyesdad
0,20758,Obesity_Type_II
1,20759,Overweight_Level_I
2,20760,Obesity_Type_III
3,20761,Obesity_Type_II
4,20762,Obesity_Type_III


MODEL 3 — Naive Bayes

In [31]:
#fit
nb_model = GaussianNB()

nb_model.fit(X_train_processed, y_train)

,priors,None
,var_smoothing,1e-09


In [32]:
#predict
nb_pred = nb_model.predict(X_val_processed)

In [33]:
#accuracy
print("Naive Bayes Accuracy:")
print(accuracy_score(y_val, nb_pred))

Naive Bayes Accuracy:
0.5859826589595376


In [35]:
#Classification Report
nb_test_pred = nb_model.predict(X_test_processed)

nb_labels = label_encoder.inverse_transform(nb_test_pred)

nb_submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": nb_labels
})

nb_submission.to_csv(
    "obesity_nb_submission.csv",
    index=False
)

nb_submission.head()

,id,NObeyesdad
0,20758,Obesity_Type_II
1,20759,Obesity_Type_I
2,20760,Obesity_Type_III
3,20761,Obesity_Type_II
4,20762,Obesity_Type_III


MODEL 4 — Support Vector Machine (SVM)

In [36]:
#model
svm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        random_state=42
    ))
])

In [37]:
#train
svm_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [38]:
#predict
svm_pred = svm_model.predict(X_val)

In [39]:
#accuracy
print("SVM Accuracy:")
print(accuracy_score(y_val, svm_pred))

SVM Accuracy:
0.8810211946050096


In [40]:
print(classification_report(
    y_val,
    svm_pred,
    target_names=label_encoder.classes_
))

                     precision    recall  f1-score   support

Insufficient_Weight       0.90      0.93      0.92       505
      Normal_Weight       0.86      0.83      0.84       617
     Obesity_Type_I       0.85      0.88      0.87       582
    Obesity_Type_II       0.96      0.97      0.96       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.74      0.70      0.72       485
Overweight_Level_II       0.77      0.77      0.77       504

           accuracy                           0.88      4152
          macro avg       0.87      0.87      0.87      4152
       weighted avg       0.88      0.88      0.88      4152



In [41]:
svm_test_pred = svm_model.predict(X_test)

svm_labels = label_encoder.inverse_transform(svm_test_pred)

svm_submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": svm_labels
})

svm_submission.to_csv(
    "obesity_svm_submission.csv",
    index=False
)

svm_submission.head()

,id,NObeyesdad
0,20758,Obesity_Type_II
1,20759,Overweight_Level_I
2,20760,Obesity_Type_III
3,20761,Obesity_Type_I
4,20762,Obesity_Type_III


In [42]:
#comparison
results = pd.DataFrame({
    "Model": [
        "Multinomial Logistic Regression",
        "LDA",
        "Naive Bayes",
        "SVM"
    ],
    "Accuracy": [
        accuracy_score(y_val, log_pred),
        accuracy_score(y_val, lda_pred),
        accuracy_score(y_val, nb_pred),
        accuracy_score(y_val, svm_pred)
    ]
})

results.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy
3,SVM,0.881021
0,Multinomial Logistic Regression,0.869220
1,LDA,0.822977
2,Naive Bayes,0.585983


# Final Model Performance Comparison

| Model | Accuracy |
|---|---|
| Support Vector Machine (SVM) | 88.1% |
| Multinomial Logistic Regression | 86.9% |
| Linear Discriminant Analysis (LDA) | 82.3% |
| Naive Bayes | 58.6% |

# Interpretation of Results

## Support Vector Machine (SVM)

The Support Vector Machine (SVM) model achieved the highest overall classification accuracy at approximately 88.1%. The strong performance of the SVM suggests that nonlinear relationships exist within the obesity risk dataset. By using the radial basis function (RBF) kernel, the SVM model was able to capture complex decision boundaries among the obesity categories. This indicates that obesity classification is influenced by nonlinear interactions among demographic, lifestyle, and physical health variables.

## Multinomial Logistic Regression

The multinomial logistic regression model also performed very well, achieving approximately 86.9% accuracy. This result suggests that many obesity risk categories can still be effectively separated using linear relationships among the predictors. Logistic regression demonstrated strong classification performance across most obesity classes and provided an interpretable baseline model for comparison.

## Linear Discriminant Analysis (LDA)

Linear Discriminant Analysis (LDA) achieved an accuracy of approximately 82.3%. While the model performed reasonably well, its accuracy was lower than both SVM and logistic regression. This may be due to the assumptions of LDA, including normally distributed predictors within each class and equal covariance matrices across classes. These assumptions may not fully hold in the obesity dataset.

## Naive Bayes

Naive Bayes produced the weakest performance with approximately 58.6% accuracy. The lower performance likely resulted from violations of the conditional independence assumption used by Naive Bayes. Many of the obesity-related predictors, such as weight, eating habits, physical activity, and family history, are naturally correlated with one another, reducing the effectiveness of the model.

# Overall Conclusion

Among the four classification methods evaluated, the Support Vector Machine provided the strongest predictive performance for the obesity risk dataset. The results suggest that nonlinear classification methods are highly effective for this multi-class prediction problem. Although multinomial logistic regression also performed well, the SVM model produced the highest accuracy and appears to capture the complex relationships within the dataset most effectively.